# sep04 — answer-ordered held-out eval (Kaggle T4 x2): the counterfactual detector

**Settings → Accelerator → GPU T4 x2, Internet → On.** Secrets: `HF_TOKEN` (required).
Queue AFTER the aug29 leak-free run; this one is independent of it — it scores the
**original leaked-run adapters** (from the Hub) and needs no training. Budget: ~1–1.5
GPU-hours (4 arms x 162 test puzzles, greedy, one vLLM session).

**The question**: the paper's held-out detector worked because evaluation prompts were
shuffled while the leaked GRPO training prompts were answer-ordered — so it succeeded
partly by accident. This notebook scores the SAME 162 held-out test puzzles with the
words presented in **answer-key order** (the training presentation), via
`scripts/eval_answer_ordered.py` — same reward, same parser, greedy, one session.

**Prespecified predictions** (written before running; shuffled-prompt references are the
aug27 memC control session, paper Table 2):

| Arm | Shuffled (groups 0–4 / reward) | Predicted answer-ordered |
|---|---|---|
| base | 0.1605 / 0.1654 | little change (weak copy bias, 2.6% of test groups) |
| sft | 0.3210 / 0.1910 | little change (0.3% copy rate) |
| grpo-ckpt50 | 0.4938 / 0.2519 | little change (0.2% copy rate) |
| grpo-final | 0.0247 / 0.1250 | **~4.0 / ~1.6 — a near-perfect solver** |

If the endpoint reads ~4.0/~1.6 here while reading 0.0247/0.1250 on shuffled prompts,
the copy rule is measured as a single number: the gap between two presentations of the
same puzzles. If it does NOT jump, the copy-rule diagnosis is wrong and the paper's
S7 needs rework — report either way.


In [ ]:
# Cell 1 — setup. No `huggingface-cli login` (interactive hang); HF_TOKEN env suffices.
import os, json, subprocess, time
from kaggle_secrets import UserSecretsClient
S = UserSecretsClient()
os.environ['HF_TOKEN'] = S.get_secret('HF_TOKEN')
HF_USER = 'jacksonlukas'

!git clone -b analysis/aug21 https://github.com/jacksonmlukas/connections-rl.git
%cd connections-rl
assert os.path.exists('scripts/eval_answer_ordered.py'), \
    'eval_answer_ordered.py missing -- wrong branch? needs analysis/aug21 (or main after the merge)'
!pip install -q -e . openai vllm peft accelerate
!pip show vllm | grep -E '^(Name|Version)'
r = subprocess.run(['python','-c','import connections_rl; print("import OK")'],
                   capture_output=True, text=True)
print(r.stdout, r.stderr); assert r.returncode == 0, 'not importable in subprocess'
!git clone --depth 1 https://github.com/jacksonmlukas/gvc-local.git /kaggle/working/gvc-local
os.environ['CONNECTIONS_PUZZLES'] = '/kaggle/working/gvc-local/data/puzzles/tagged_connections.json'
!make data
r = subprocess.run(['python','-c',(
    'from connections_rl.data.loader import load_puzzles;'
    'print(len(load_puzzles("data/splits/puzzles_test.json")))')], capture_output=True, text=True)
assert int(r.stdout.strip()) == 162, 'test split is not 162'
print('setup OK')


In [ ]:
# Cell 2 — PROMPT READ-BACK before any GPU spend (the lesson: verify the constructed
# object). Every consecutive quadruple of the answer-ordered user turn must be a correct
# group, and the presentation must differ from the shuffled one on these puzzles.
r = subprocess.run(['python','-c','''
import sys; sys.path.insert(0, "scripts")
from eval_answer_ordered import answer_ordered_chat
from connections_rl.data.formatting import build_chat
from connections_rl.data.loader import load_puzzles
puzzles = load_puzzles("data/splits/puzzles_test.json")[:5]
differs = 0
for p in puzzles:
    u = answer_ordered_chat(p)[1]["content"]
    words = [w.strip() for w in u.replace("Words:", "", 1).split(",")]
    assert len(words) == 16, (p.puzzle_id, len(words))
    keys = [frozenset(g.members) for g in p.groups]
    for i in range(0, 16, 4):
        assert frozenset(words[i:i+4]) in keys, ("quadruple is not a group", p.puzzle_id, i)
    if u != build_chat(p)[1]["content"]:
        differs += 1
assert differs >= 4, "answer-ordered and shuffled presentations barely differ -- inspect"
print("read-back OK: every quadruple is an answer group; presentation differs from shuffled")
'''], capture_output=True, text=True)
print(r.stdout, r.stderr); assert r.returncode == 0, 'read-back FAILED -- do not serve, inspect first'


In [ ]:
# Cell 3 — adapters from the Hub, then ONE vLLM session (same serve line as memC/aug29).
import urllib.request
from huggingface_hub import snapshot_download
snapshot_download(f'{HF_USER}/connections-rl-sft-7b', local_dir='adapters/sft-7b',
                  token=os.environ['HF_TOKEN'])
snapshot_download(f'{HF_USER}/connections-rl-grpo-7b-ckpt', local_dir='adapters/grpo-7b-ckpt',
                  allow_patterns='checkpoint-50/*', token=os.environ['HF_TOKEN'])
snapshot_download(f'{HF_USER}/connections-rl-grpo-7b', local_dir='adapters/grpo-7b',
                  token=os.environ['HF_TOKEN'])
assert os.path.exists('adapters/sft-7b/adapter_config.json'), 'SFT adapter incomplete'
assert os.path.isdir('adapters/grpo-7b-ckpt/checkpoint-50'), 'ckpt-50 missing from ckpt repo'

mods = ['connections-rl-sft-7b=adapters/sft-7b',
        'connections-rl-grpo-7b-ckpt50=adapters/grpo-7b-ckpt/checkpoint-50',
        'connections-rl-grpo-7b=adapters/grpo-7b']
proc = subprocess.Popen(
    'vllm serve Qwen/Qwen2.5-7B-Instruct --dtype half --tensor-parallel-size 2 '
    '--enable-lora --enforce-eager --max-lora-rank 16 --max-model-len 2048 '
    '--gpu-memory-utilization 0.85 --lora-modules ' + ' '.join(mods),
    shell=True, stdout=open('/kaggle/working/vllm.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(150):
    try:
        urllib.request.urlopen('http://localhost:8000/health'); print('vLLM ready'); break
    except Exception:
        time.sleep(10)
else:
    raise RuntimeError('vLLM failed to come up -- see /kaggle/working/vllm.log')
!mkdir -p results-analysis/sep04
!pip show vllm | grep -E '^(Name|Version)' | tee results-analysis/sep04/answer-ordered-session-versions.txt


In [ ]:
# Cell 4 — the eval: all four arms, answer-ordered prompts, one session, greedy.
ARMS = {'base': 'Qwen/Qwen2.5-7B-Instruct',
        'sft': 'connections-rl-sft-7b',
        'grpo-ckpt50': 'connections-rl-grpo-7b-ckpt50',
        'grpo-final': 'connections-rl-grpo-7b'}
for arm, model in ARMS.items():
    t0 = time.time()
    r = subprocess.run(['python', 'scripts/eval_answer_ordered.py',
                        '--model', model, '--arm', arm,
                        '--out', 'results-analysis/sep04/answer-ordered'])
    print(f'{arm}: exit {r.returncode} in {(time.time()-t0)/60:.0f} min')
    assert r.returncode == 0, f'{arm} failed'


In [ ]:
# Cell 5 — verdict against the prespecified table. Shuffled references are the aug27
# memC control session (paper Table 2), hardcoded from its metrics.json artifacts.
SHUFFLED = {'base':        {'groups': 0.1605, 'reward': 0.1654},
            'sft':         {'groups': 0.3210, 'reward': 0.1910},
            'grpo-ckpt50': {'groups': 0.4938, 'reward': 0.2519},
            'grpo-final':  {'groups': 0.0247, 'reward': 0.1250}}
summary = {'session': 'sep04 answer-ordered session', 'decoding': 'greedy T=0.0',
           'presentation': 'answer-key order (the leaked training presentation)',
           'n': 162, 'arms': {}}
hdr = (f"{'arm':<14} {'AO groups(0-4)':<24} {'AO reward':<22} "
       f"{'shuf groups':<12} {'shuf reward':<12} delta(reward)")
print(hdr); print('-' * len(hdr))
for arm in ARMS:
    m = json.load(open(f'results-analysis/sep04/answer-ordered/{arm}/metrics.json'))
    s = m['summary']['OVERALL']
    g, w = s['groups_correct'], s['reward']
    ref = SHUFFLED[arm]
    summary['arms'][arm] = {'groups_mean_ci': g, 'reward_mean_ci': w,
                            'slots': round(g[0] * 162), 'slots_of': 648,
                            'invalid_rate_ci': s['invalid_rate'],
                            'shuffled_reference_memC': ref}
    print(f"{arm:<14} {g[0]:.4f} [{g[1]:.3f},{g[2]:.3f}]   "
          f"{w[0]:.4f} [{w[1]:.3f},{w[2]:.3f}]   "
          f"{ref['groups']:<12.4f} {ref['reward']:<12.4f} {w[0]-ref['reward']:+.4f}")
fin = summary['arms']['grpo-final']
print()
if fin['reward_mean_ci'][0] > 1.4 and fin['groups_mean_ci'][0] > 3.5:
    print('PRESPECIFIED PREDICTION CONFIRMED: the endpoint is a near-perfect solver of the')
    print('training presentation of the SAME puzzles it near-zeroes when shuffled.')
else:
    print('PREDICTION NOT CONFIRMED: the endpoint did not read ~1.6/~4.0 on answer-ordered')
    print('prompts. The copy-rule diagnosis needs rework; report this as measured.')
json.dump(summary, open('results-analysis/sep04/answer_ordered_summary.json', 'w'), indent=1)
print('wrote results-analysis/sep04/answer_ordered_summary.json')


In [ ]:
# Cell 6 — persist.
!zip -qr /kaggle/working/sep04-answer-ordered-outputs.zip results-analysis/sep04
print('zip ready: /kaggle/working/sep04-answer-ordered-outputs.zip')
try:
    from huggingface_hub import HfApi
    HfApi(token=os.environ['HF_TOKEN']).upload_folder(
        folder_path='results-analysis/sep04',
        repo_id=f'{HF_USER}/connections-rl-results', repo_type='dataset',
        path_in_repo='sep04')
    print('Hub upload OK -> connections-rl-results/sep04')
except Exception as e:
    print('Hub upload skipped/failed (fine -- use the zip):', e)
